# INTRODUCTION

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">

This Exploratory Data Analysis (EDA) project analyzes a comprehensive US Real Estate dataset scraped from Realtor.com, containing over 2.2 million listing records across 10 key features. The primary focus is to examine the economic and physical factors driving real estate listing prices across different US states, cities, and zip codes. By applying data auditing, cleaning, and descriptive visualization techniques, this study aims to resolve data quality challenges like heavy skewness and missing values, while unearthing key market patterns to inform future predictive modeling and real estate insights.


</div>

# OBJECTIVES:

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">

<ul>
<li>Data Preprocessing: Clean the dataset by handling missing values, removing duplicates, filtering extreme outliers, and reformatting date/categorical fields.

<li>Univariate Analysis: Evaluate the overall distribution, skewness, and central tendencies of key numerical features such as price, house_size, and acre_lot.

<li>Geographical Insights: Analyze price variations and price-per-square-foot trends across different states, cities, and zip codes to identify high-value versus affordable markets.

<li>Correlation & Relationship Mapping: Determine the strength of relationships between physical property features (bedrooms, bathrooms, living area) and final listing prices.

<li>Temporal Trend Analysis: Explore historical market behavior using prev_sold_date to observe sales activity patterns over time.

<li>Feature Engineering: Create derived metrics like price_per_sqft and total_rooms to prepare a clean, structured baseline for future machine learning models.

</div>

<div
>

# 1. Getting Ready with Datasets


In [ ]:
import numpy as np
import pandas as pd
import polars as pl

import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
data = pd.read_parquet('/home/kartika/Datasets/brazilianolist.csv', engine='pyarrow')

for col in data.select_dtypes(include = ['object']).columns :
    data[col] = data[col].astype('category')

for col in data.select_dtypes(include = ['int', 'float']).columns :
    data[col] = pd.to_numeric(data[col], downcast = 'integer' if 'int' in str(data[col].dtype) else 'float')



print(f"Memory Usage: {data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


<div>

<div>

# 2. Cleaning and Wraggling

In [ ]:
data.head(5)


In [ ]:
data.info()

In [ ]:
data['price'] = data['price'].fillna(
    data.groupby('city', observed = True)['price'].transform('median')
)

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify"><1% values are missing, imputed them with median price, grouped by city.

<div>

In [ ]:

bins = list(range(0, 10000, 500)) + [np.inf]
data['sqft_bin'] = pd.cut(data['house_size'], bins=bins)


data['bed'] = data['bed'].fillna(
    data.groupby('sqft_bin', observed=True)['bed'].transform('median')
)

data['bath'] = data['bath'].fillna(
    data.groupby('sqft_bin', observed=True)['bath'].transform('median')
)

data['bed'] = data['bed'].fillna(data['bed'].median())
data['bath'] = data['bath'].fillna(data['bath'].median())

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
Around 22% values are missing in bed field and 23% values are missing in bath field, imputed using house size bins.

<div>

In [ ]:
data['acre_lot'] = data['acre_lot'].fillna(
    data.groupby('city', observed=True)['acre_lot'].transform('median')
)


if 'state' in data.columns:
    data['acre_lot'] = data['acre_lot'].fillna(
        data.groupby('state', observed=True)['acre_lot'].transform('median')
    )

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">Around 15% values are missing in acre_lot, so imputed via grouped city.

<div>

In [ ]:
data['city'] = data['city'].dropna()

In [ ]:
data['state'] = data['city'].dropna()

In [ ]:
data['zip_code'] = data['zip_code'].dropna()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
Since, <1% values are missing, dropped them.

<div>

In [ ]:

data['house_size_was_missing'] = data['house_size'].isna().astype(int)

# If hosue_size and Bed/Bath both are missing
land_mask = (
    data['house_size'].isna() 
    & (data['bed'].isna() | (data['bed'] == 0)) 
    & (data['bath'].isna() | (data['bath'] == 0))
)
data.loc[land_mask, 'house_size'] = 0




# Imputation via group median
data['house_size'] = data['house_size'].fillna(
    data.groupby(['city', 'bed', 'bath'], observed=True)['house_size'].transform('median')
)



data['house_size'] = data['house_size'].fillna(
    data.groupby(['city', 'bed'], observed=True)['house_size'].transform('median')
)



data['house_size'] = data['house_size'].fillna(
    data.groupby(['city', 'bath'], observed=True)['house_size'].transform('median')
)

data['house_size'] = data['house_size'].fillna(
    data.groupby('city', observed=True)['house_size'].transform('median')
)


data['house_size'] = data['house_size'].fillna(data['house_size'].median())

<div>

In [2]:
# data['has_prev_sale'] = data['prev_sold_date'].notna()

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">Created new field(has_prev_sale) from prev_sold_date.